# Module 10 Part 2: SQL for Your Project - Your Own Neon or DuckDB

**BAN 6003: Data Management and Analytics Integration**

This notebook is a **guide, not a graded lab**. It shows the two ways to meet the SQL requirement of the course project. Read Section 1, choose one path, and work through that part. You may try both.

> Project rule: your project notebook must include at least one SQL query on your own project data, using **Path A** or **Path B**.

## 1. Choose Your Path

| | Path A: your own Neon database | Path B: DuckDB |
|---|---|---|
| What it is | A free Postgres database in the cloud that **you** own | A small database engine that runs inside Python |
| Account needed | Yes, a free Neon account | No |
| Where your data lives | You upload your CSV files or DataFrames to your database | Your DataFrames and CSV files stay where they are |
| Connection string | Yes, saved in `.env` as `PROJECT_DATABASE_URL` | No |
| SQL you write | Postgres SQL, as in Modules 9-10 | Almost the same SQL (see Module 10, Section 11) |
| Good for | Practicing a real database workflow: create, upload, query | The fastest way to add SQL checks to your project |

Both paths count the same for the project. Choose Path A if you want hands-on practice with your own database. Choose Path B if you want the simplest setup.

**How to use this notebook in your project.** Your project data is in your **project** repository, not in this one. Cells whose first line says `# COPY` are written so you can copy them into your project notebook without changes. Practice here first with a small practice file, then copy the cells into your project notebook and run them there.

## 2. Setup

This cell loads the packages and creates a small practice CSV file, `data/processed/practice_planes.csv`, with one row per aircraft (3,322 rows). The rest of this notebook uses it as stand-in project data.

In [ ]:
# Codespaces: these packages are installed from requirements.txt.
# Local only: if an import fails, uncomment and run this line:
# %pip install pandas sqlalchemy psycopg2-binary "psycopg[binary]" python-dotenv duckdb nycflights13

from pathlib import Path

import pandas as pd
from nycflights13 import planes

# Find the top folder of the repository (the folder that contains README.md).
repo_root = Path.cwd()
if not (repo_root / "README.md").exists() and (repo_root.parent / "README.md").exists():
    repo_root = repo_root.parent

practice_csv = repo_root / "data" / "processed" / "practice_planes.csv"
practice_csv.parent.mkdir(parents=True, exist_ok=True)
planes.to_csv(practice_csv, index=False)
print("Practice file:", practice_csv.relative_to(repo_root), "| rows:", len(planes))


## 3. Path A: Your Own Neon Database

In Modules 9-10 you **read** from the course database. Here you create your **own** database, so you can also **write**: you upload your project tables and then query them with SQL.

Path A has four steps:

1. Create a free Neon project and copy its connection string (Section 3.1).
2. Save the string in `.env` as `PROJECT_DATABASE_URL` and connect (Section 3.2).
3. Upload your CSV files or DataFrames (Section 3.3).
4. Query your tables with SQL and save the results (Section 3.4).

If you chose Path B, you can skip to Section 4. The Path A cells below still run without errors: they print a message and skip the database steps.

### 3.1 Create your Neon project

1. Go to [neon.com](https://neon.com) and sign up for a free account. Signing up with your GitHub account is the easiest option.
2. Create a new project. Give it a clear name such as `ban6003-project`. For everything else, the default choices are fine.
3. In the Neon Console, click **Connect**. Neon shows a connection string that starts with `postgresql://`. Copy it.

Good to know about the free plan:

- Each project can store up to 1 GB, which is much more than the course project data needs.
- The database goes to sleep after 5 minutes without queries. The first query after a break can take a few seconds while it wakes up. This is normal.

**Your connection string contains your password and gives full access to your database.** Keep it only in `.env`. Never paste it into a notebook, a GitHub file, a Canvas post, or a chat. If it is ever exposed, reset the password in the Neon Console (Roles) and update `.env`.

### 3.2 Save the connection string and connect

Run the next cell. The first time, it adds an empty line `PROJECT_DATABASE_URL=` to your `.env` file (the file is created if it does not exist) and tells you what to do. Open `.env`, paste your **own** connection string right after `PROJECT_DATABASE_URL=` (no spaces, no quotes), save the file, and run the cell again.

Use the name `PROJECT_DATABASE_URL` only for your own database. `DATABASE_URL` stays the read-only course database.

The cell also creates the folder `outputs/sql_results`. Section 3.4 explains why.

**In your project:** copy this cell into your project notebook. Your project repository has its own `.env` file (Git ignores it there too), so paste your connection string there as well.

In [ ]:
# COPY - Path A, cell 1 of 3: connect to your own Neon database
import os
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine, text

# Find the top folder of the repository (the folder that contains README.md).
repo_root = Path.cwd()
if not (repo_root / "README.md").exists() and (repo_root.parent / "README.md").exists():
    repo_root = repo_root.parent

# Add an empty PROJECT_DATABASE_URL line to .env the first time. Git ignores .env.
env_path = repo_root / ".env"
env_text = env_path.read_text() if env_path.exists() else ""
if "PROJECT_DATABASE_URL" not in env_text:
    if env_text and not env_text.endswith("\n"):
        env_text += "\n"
    env_text += ("# Your own Neon database (Path A). Paste your connection string after the = sign.\n"
                 "PROJECT_DATABASE_URL=\n")
    env_path.write_text(env_text)
    print("Added an empty PROJECT_DATABASE_URL line to .env")

load_dotenv(env_path, override=True)
project_url = os.getenv("PROJECT_DATABASE_URL")

# Saved query results go here, so the notebook also runs without your .env (see Section 3.4).
SQL_RESULTS_DIR = repo_root / "outputs" / "sql_results"
SQL_RESULTS_DIR.mkdir(parents=True, exist_ok=True)

if project_url:
    project_engine = create_engine(project_url)
    with project_engine.connect() as conn:
        conn.execute(text("SELECT 1;"))
    print("Connected to your own database. For safety, we do not print the connection string.")
else:
    project_engine = None
    print("PROJECT_DATABASE_URL is empty, so the database steps will be skipped.")
    print("Path A: open .env, paste your own Neon connection string after PROJECT_DATABASE_URL=, save, and run this cell again.")


### 3.3 Upload your data

`upload_table()` copies one table into your database:

- give it a **CSV file path** to upload a raw data file, or
- give it a **DataFrame** to upload a table you cleaned or built in Pandas, for example your ABT.

It writes the table with `to_sql(..., if_exists="replace")`, so running it again replaces the old copy instead of adding duplicate rows. Then it compares the row count in the database with the row count in Python. The two numbers should be equal.

Table names: use lowercase letters, numbers, and underscores, for example `movie_abt` or `order_items`. Then you never need quotes around the name in SQL.

**Column names with capital letters need double quotes in Postgres.** The upload keeps your column names exactly as they are in Pandas. Postgres turns unquoted names into lowercase, so a column such as `movieId` must be written as `"movieId"` in your SQL, for example `COUNT(DISTINCT "movieId")`. Without the quotes, Postgres looks for `movieid` and reports that the column does not exist. (DuckDB in Path B does not have this problem.)

In [ ]:
# COPY - Path A, cell 2 of 3: upload a CSV file or a DataFrame to your database
def upload_table(source, table_name):
    """Upload a CSV file path or a DataFrame to your own database and check the row count."""
    if project_engine is None:
        print(f"Skipped uploading {table_name}: no PROJECT_DATABASE_URL in .env.")
        return
    df = source if isinstance(source, pd.DataFrame) else pd.read_csv(source, low_memory=False)
    df.to_sql(table_name, project_engine, if_exists="replace", index=False, chunksize=1000, method="multi")
    with project_engine.connect() as conn:
        db_rows = conn.execute(text(f"SELECT COUNT(*) FROM {table_name};")).scalar()
    status = "OK" if db_rows == len(df) else "CHECK: the row counts differ"
    print(f"{table_name}: {len(df):,} rows in Python, {db_rows:,} rows in the database. {status}")


# Fill in your own files. One line per table: upload_table(path_or_dataframe, "table_name")
# In your project, use your data files, for example:
#   upload_table(repo_root / "data" / "ratings.csv", "ratings")
#   upload_table(abt, "movie_abt")      # a DataFrame you built in Pandas
upload_table(practice_csv, "practice_planes")


You only need to upload a table again when its data changes. If your ABT changes, run `upload_table(abt, "your_abt_name")` again to replace the copy in the database.

In your project, `practice_csv` does not exist: replace the last line with your own `upload_table(...)` lines.

### 3.4 Query your tables, and keep the notebook runnable for the instructor

Your `.env` file is not on GitHub, so the instructor cannot connect to your database. To keep your project notebook runnable, `run_project_sql()` works in two modes:

- **With** `PROJECT_DATABASE_URL`: it runs the query on your database and saves the result as a CSV file in `outputs/sql_results`.
- **Without** it (for example, when the instructor runs your notebook): it shows the saved result instead and says so.

So: run every query once yourself, then **commit and push the `outputs/sql_results` files** together with your notebook. Never commit `.env`.

In [ ]:
# COPY - Path A, cell 3 of 3: run SQL on your database and save the result
def run_project_sql(query, result_name):
    """Run SQL on your own database and save the result. Without a connection, show the saved result."""
    result_path = SQL_RESULTS_DIR / f"{result_name}.csv"
    if project_engine is not None:
        result = pd.read_sql(text(query), project_engine)
        result.to_csv(result_path, index=False)
        print(f"Ran on your database. Saved the result to {result_path.relative_to(repo_root)}")
        return result
    if result_path.exists():
        print(f"No database connection here, so this is the saved result from {result_path.relative_to(repo_root)}")
        return pd.read_csv(result_path)
    print(f"Skipped {result_name}: no PROJECT_DATABASE_URL and no saved result yet.")
    return pd.DataFrame()


Example: a key check and a grouped summary on the practice table. In your project, replace the table and column names with yours, and give each query its own `result_name`.

In [ ]:
# Key check: does every row have its own tailnum?
key_check = run_project_sql("""
SELECT
    COUNT(*) AS row_count,
    COUNT(DISTINCT tailnum) AS distinct_tailnum
FROM practice_planes;
""", "practice_key_check")
key_check


In [ ]:
# Grouped summary: the five most common manufacturers
manufacturer_summary = run_project_sql("""
SELECT
    manufacturer,
    COUNT(*) AS plane_count,
    ROUND(AVG(seats)::numeric, 1)::float AS avg_seats
FROM practice_planes
GROUP BY manufacturer
ORDER BY plane_count DESC
LIMIT 5;
""", "practice_manufacturer_summary")
manufacturer_summary


If you connected, the key check shows 3,322 rows and 3,322 distinct tail numbers: one row per aircraft. If you did not connect and have no saved results, these cells print a skip message. That is fine if you chose Path B.

**Path A checklist for your project notebook:**

1. Copy the three `# COPY` cells into your project notebook.
2. Paste your own connection string into the project repository's `.env` as `PROJECT_DATABASE_URL`.
3. Replace the practice upload line with your own `upload_table(...)` lines and run them.
4. Write at least one SQL query with `run_project_sql(query, "a_clear_name")` that supports your integration or validation work.
5. Under each query, write one or two sentences on what the result tells you.
6. Commit and push the notebook and the files in `outputs/sql_results`. Check that `.env` is **not** in the commit.

## 4. Path B: DuckDB on Your DataFrames and CSV Files

DuckDB runs SQL directly on the DataFrames in your notebook and the CSV files in your repository, with no account and no connection string. Module 10, Section 11 teaches it with the flights data. This section shows the same pattern on project-style data.

Two ways to name a table in DuckDB:

- a DataFrame: use its variable name, for example `FROM abt`;
- a CSV file: use `read_csv('path/to/file.csv')`.

Remember: in DuckDB, `ROUND(AVG(column), 1)` already returns normal numbers, so you do not need `::numeric` and `::float`.

Because DuckDB runs inside the notebook, the instructor can rerun your queries directly. You do not need to save the results.

In [ ]:
# COPY - Path B: DuckDB setup
import duckdb

print("DuckDB version:", duckdb.__version__)


Example 1: query a DataFrame. Here the DataFrame is `practice_planes_df`; in your project, use your own DataFrame name, such as `abt`.

In [ ]:
practice_planes_df = pd.read_csv(practice_csv)

# Key check on a DataFrame: compare all rows with distinct key values
duckdb.sql("""
SELECT
    COUNT(*) AS row_count,
    COUNT(DISTINCT tailnum) AS distinct_tailnum
FROM practice_planes_df;
""").df()


Example 2: query a CSV file directly, without loading it into Pandas first. `as_posix()` writes the path with forward slashes, which DuckDB accepts on every operating system.

In [ ]:
duckdb.sql(f"""
SELECT
    manufacturer,
    COUNT(*) AS plane_count,
    ROUND(AVG(seats), 1) AS avg_seats
FROM read_csv('{practice_csv.as_posix()}')
GROUP BY manufacturer
ORDER BY plane_count DESC
LIMIT 5;
""").df()


Both examples give the same answers as Path A: 3,322 rows and 3,322 distinct tail numbers, and the same five manufacturers.

**Path B checklist for your project notebook:**

1. Copy the `# COPY - Path B` cell into your project notebook. The Movies and Olist starter notebooks already have a "SQL Validation with DuckDB" section that you can build on.
2. Write at least one DuckDB query on your own DataFrames or CSV files that supports your integration or validation work.
3. Under each query, write one or two sentences on what the result tells you.

## 5. What a Good Project SQL Query Looks Like

With either path, a good query **supports your data work**. It is not a separate exercise. Strong examples:

- **Key check:** `COUNT(*)` vs `COUNT(DISTINCT key)` on your ABT, to show one row per unit of analysis.
- **Row count check:** row counts before and after a join.
- **Rebuild one step in SQL:** redo one aggregation of your ABT in SQL and compare the result with your Pandas result.
- **Lower-level table summary:** aggregate a detail table (for example, ratings per movie or items per order) before joining it.

For each query, show the SQL, its result, and one or two sentences on what the result tells you.